# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Hello World

In [ ]:
# First, let's print a simple message to ensure our environment is set up correctly.
print("Hello World")

## 2. Initial Setup

In [1]:
#!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Free GPU Memory (GB): 39.3936


In [ ]:
!nvidia-smi

In [ ]:
import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface/"

os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH

import torch

torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Code formatting and linting

!black notebooks/Llama-3-8B-quant.ipynb
!pylint notebooks/Llama-3-8B-quant.ipynb

# Checking CUDA availability
import torch

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
# HuggingFace authentication
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token)

## 3. Loading Models

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# device = "cpu"
device = "cuda"

# model_name = "EleutherAI/gpt-neo-125m"  # Lightweight model for debugging purposes
model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v0.1"  # Small enough to run on a gpu_gtx1080.

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device, torch_dtype=torch.float16)

# Initialize model with empty weights
# from accelerate import init_empty_weights, load_checkpoint_and_dispatch
# with init_empty_weights():
#     model = AutoModelForCausalLM.from_config(torch_dtype=torch.float16, config=AutoModelForCausalLM.from_pretrained(model_name).config)
# model = load_checkpoint_and_dispatch(model, model_name, device_map="auto", offload_folder="offload")

model.NAME = model_name

if tokenizer.model_max_length > 1e6:
  tokenizer.model_max_length = 2048

print(f"Model Memory Footprint: {(model.get_memory_footprint() / (1024 ** 3)):.2f} GB")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
# Example inference

from transformers import AutoTokenizer
import transformers 
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)
pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

prompt = "What famous tower is in Paris?"
formatted_prompt = (
    f"### Human: {prompt}### Assistant:"
)


sequences = pipeline(
    formatted_prompt,
    do_sample=True,
    top_k=50,
    top_p = 0.7,
    num_return_sequences=1,
    repetition_penalty=1.1,
    max_new_tokens=500,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")


In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

## 4. Loading Datasets

### 4.1. WikiText

In [ ]:
# Initialize the datamodule
import os
from src.data.WikiTextDataModule import WikiTextDataModule

# wikitext_sequence_length = tokenizer.model_max_length
wikitext_sequence_length = 10
wikitext_sequence_length = 1024
wikitext_stride = 512
wikitext_batch_size = 1  # Just use batch size 1 for this project
#wikitext_batch_size = 16
#wikitext_batch_size = 64
wikitext_seed = 1

wikitext_data_module = WikiTextDataModule(
  directory_dataset=os.getcwd(),
  batch_size=wikitext_batch_size,
  sequence_length=wikitext_sequence_length,
  stride=wikitext_stride,
  tokenizer_name=model_name,
  seed=wikitext_seed
)

split = "validation"

dataloader_from_split = lambda data_module: {
  "train": data_module.train_dataloader(),
  "validation": data_module.val_dataloader(),
  "test": data_module.test_dataloader()
}
wikitext_dataloader = dataloader_from_split(wikitext_data_module)[split]

# Print properties
print("Length of datasets:", len(wikitext_data_module.train_dataset), len(wikitext_data_module.val_dataset), len(wikitext_data_module.test_dataset))
print("Total number of tokens for each dataset:", len(wikitext_data_module.train_dataset), len(wikitext_data_module.val_dataset), len(wikitext_data_module.test_dataset))
print("Total number of tokens for the val dataset:", sum([len(data_string) for data_string in wikitext_data_module.val_dataset['text']]))

total_string = "".join([data_string for data_string in wikitext_data_module.val_dataset['text']])
total_string_len = len(total_string)
tokenized_string = tokenizer.encode(total_string, return_tensors="pt")

print(f"Length of total dataset: {total_string_len}")
print(f"Length of tokenized dataset: {len(tokenized_string[0])}")
print(f"Tokenizer compression rate: {(100 * len(tokenized_string[0]) / total_string_len):.2f}%.")

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

dataset_size = len(wikitext_dataloader)
print(f"Number of batches in dataloader: {dataset_size}")
for i, (data, target) in enumerate(wikitext_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")


In [ ]:
decoded = tokenizer.decode(encoded['input_ids'])
print(decoded)

### 4.2. OpenAssistant

In [ ]:
# Initialize the datamodule
import os
from src.data.OpenAssistantDataModule import OpenAssistantDataModule

directory_dataset = os.getcwd()
oasst_batch_size = 1  # Just use batch size 1 for this project
# oasst_batch_size = 16
# oasst_batch_size = 64
oasst_sequence_length = 512  # Maximum sequence length - use the default value
oasst_seed = 1

# Data Module
oasst_data_module = OpenAssistantDataModule(
  directory_dataset=directory_dataset,
  batch_size=oasst_batch_size,
  sequence_length=oasst_sequence_length,
  tokenizer_name=model_name,
  seed=oasst_seed
)

# Data Loader
# oasst_dataloader = oasst_data_module.train_dataloader()
oasst_dataloader = oasst_data_module.val_dataloader()

print(f"Length of datasets:", len(oasst_data_module.train_dataset), len(oasst_data_module.val_dataset))

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

oasst_dataset_size = len(oasst_dataloader)
print(f"Number of batches in train_dataloader: {dataset_size}")
for i, (data, target) in enumerate(oasst_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")

## 5. Quantization

### 5.1. BitsAndBytes

#### 5.1.1 BitsAndBytes 8-bit

In [ ]:
# BNB Config 8-bit

from transformers import BitsAndBytesConfig

from src import MODEL_SAVE_PATH

# Define the device and model name
device = "cuda"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# BnB Quantization Configurations
bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    load_in_4bit=False,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
    llm_int8_has_fp16_weight=False,
)

# Save path
import os
bnb_8bit_model_name = f"{model_name.split('/')[1]}-bnb-8bit"
bnb_8bit_model_path = os.path.join(MODEL_SAVE_PATH, bnb_8bit_model_name)
os.makedirs(bnb_8bit_model_path, exist_ok=True)

In [ ]:
# Quantization 8-bit

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model_bnb_8bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_8bit, 
    torch_dtype=torch.float32,
    device_map=device
)
model_bnb_8bit.NAME = bnb_8bit_model_name

print(f"8-bit BNB Model Memory Footprint: {(model_bnb_8bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")

from accelerate import Accelerator
accelerate = Accelerator()
accelerate.save_model(model_bnb_8bit, bnb_8bit_model_path)

print(f"8-bit BnB model saved at: {bnb_8bit_model_path}")

# from src.models.utils_llm import calculate_model_size
# calculate_model_size(bnb_8bit_model_path)
# from src.models.utils_llm import print_gpu_utilization
# print_gpu_utilization()

#### 5.1.2 BitsAndBytes 4-bit

In [ ]:
# BNB Config 4-bit

import torch
from transformers import BitsAndBytesConfig

from src import MODEL_SAVE_PATH

# Define the device and model name
device = "cuda"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config_4bit = BitsAndBytesConfig(
    load_in_8bit=False,
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="fp4",
    bnb_4bit_use_double_quant=False,
)

# Save path
import os
bnb_4bit_model_name = f"{model_name.split('/')[1]}-bnb-4bit"
bnb_4bit_model_path = os.path.join(MODEL_SAVE_PATH, bnb_4bit_model_name)

In [ ]:
# Quantization 4-bit

from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model_bnb_4bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_4bit, 
    torch_dtype=torch.float32,
    device_map=device
)
model_bnb_4bit.NAME = bnb_4bit_model_name

print(f"4-bit BNB Model Memory Footprint: {(model_bnb_4bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")

from accelerate import Accelerator
accelerate = Accelerator()
accelerate.save_model(model_bnb_4bit, bnb_4bit_model_path)

print(f"4-bit BnB model saved at: {bnb_4bit_model_path}")

# from src.models.utils_llm import calculate_model_size
# calculate_model_size(bnb_4bit_model_path)
# from src.models.utils_llm import print_gpu_utilization
# print_gpu_utilization()

### 5.2 AWQ

In [ ]:
# AWQ Config
awq_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

# AWQ Calibration Split
awq_calib_split = "validation"

# Save path
import os
awq_model_name = f"{model_name.split('/')[1]}-awq"
awq_model_path = os.path.join(MODEL_SAVE_PATH, awq_model_name)

# Define the device and model name
device = "cuda"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [ ]:
# AWQ Quantization

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

awq_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
awq_model = AutoAWQForCausalLM.from_pretrained(
    model_name,
    device_map=device
)

# Quantize with wikitext validation as calibration data
awq_model.quantize(
    tokenizer=tokenizer,
    quant_config=awq_config,
    calib_data=wikitext_dataset,  # Pass the loaded validation dataset here
)
awq_model.NAME = awq_model_name

# Save quantized model
awq_model.save_quantized(awq_model_path)
awq_tokenizer.save_pretrained(awq_model_path)

print(f'Model is quantized and saved at "{awq_model_path}"')

# from src.models.utils_llm import calculate_model_size
# calculate_model_size(awq_model_path)
# from src.models.utils_llm import print_gpu_utilization
# print_gpu_utilization()

In [ ]:
# Load model and generate text
awq_model_path = "TinyLlama-1.1B-Chat-v1.0-awq"

awq_tokenizer = AutoTokenizer.from_pretrained(awq_model_path)
awq_model = AutoAWQForCausalLM.from_pretrained(
    awq_model_path,
    trust_remote_code=True,
)

# Generate text
prompt = "What is the weather like today?"
input_ids = awq_tokenizer.encode(prompt, return_tensors="pt")  # Convert text to tensors

output = awq_model.generate(input_ids)
generated_text = awq_tokenizer.decode(output[0], skip_special_tokens=True)

### 5.3 HQQ

In [ ]:
# HQQ Config

from transformers import AutoTokenizer

# Define the model name and tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

### 5.3.1 Option 1: All linear layers will use the same quantization config

In [ ]:
import torch
from transformers import AutoModelForCausalLM, HqqConfig

# Option 1: All linear layers will use the same quantization config
quant_config_same = HqqConfig(
    nbits=8, 
    group_size=64, 
    quant_zero=False, 
    quant_scale=False, 
    axis=0  # Default value
)

# Quantize the model with the same config for all linear layers
model_same = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
    quantization_config=quant_config_same
)

# Save the quantized model with the same config for all layers
model_same_path = f"{model_name}-hqq-same"
model_same.save_pretrained(model_same_path)
tokenizer.save_pretrained(model_same_path)
print(f"Model with the same quantization config saved at '{model_same_path}'")

from src.models.utils_llm import calculate_model_size
print(f"Model size (same config): {calculate_model_size(model_same_path)}")

### 5.3.2. Option 2: Different configs for specific layers

In [ ]:
import torch
from transformers import AutoModelForCausalLM, HqqConfig

# Option 2: Different configs for specific layers
q4_config = {'nbits': 4, 'group_size': 64, 'quant_zero': False, 'quant_scale': False}
q3_config = {'nbits': 3, 'group_size': 32, 'quant_zero': False, 'quant_scale': False}

quant_config_dynamic = HqqConfig(dynamic_config={
    'self_attn.q_proj': q4_config,
    'self_attn.k_proj': q4_config,
    'self_attn.v_proj': q4_config,
    'self_attn.o_proj': q4_config,
    'mlp.gate_proj': q3_config,
    'mlp.up_proj': q3_config,
    'mlp.down_proj': q3_config,
})

# Quantize the model with different configs for specific layers
model_dynamic = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
    quantization_config=quant_config_dynamic
)

# Save the quantized model with different configs for specific layers
model_dynamic_path = f"{model_name}-hqq-dynamic"
model_dynamic.save_pretrained(model_dynamic_path)
tokenizer.save_pretrained(model_dynamic_path)
print(f"Model with dynamic quantization config saved at '{model_dynamic_path}'")

from src.models.utils_llm import calculate_model_size
print(f"Model size (dynamic config): {calculate_model_size(model_dynamic_path)}")

## 6. Evaluation

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'


### 6.1. Perplexity

In [ ]:
import torch
import torchmetrics
import tqdm
from torch.cuda.amp import autocast

# with torch.no_grad():
#     torch.cuda.empty_cache()
    
wikitext_dataloader = wikitext_data_module.test_dataloader()

model.eval()
metric = torchmetrics.text.Perplexity(ignore_index=-100).to(device)  # -100 is the padding token.

for i, (x, y) in enumerate(wikitext_dataloader):
    print(f"Processing batch {i}")
    x, y = x.to(device), y.to(device)
    
    with torch.no_grad() and autocast():
        outputs = model(x)
        logits = outputs.logits
        !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
        
        # Metric on current batch
        perplexity = metric(logits.float(), y)   
        print(f"Perplexity: {perplexity:.2f}")
        # del outputs, logits, perplexity
        # torch.cuda.empty_cache()

# Metric on all batches using custom accumulation
perplexity = metric.compute()
print(perplexity.item())

1016.52s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Free GPU Memory (GB): 7.94922
Perplexity: 6.99
Processing batch 39


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
1022.16s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Free GPU Memory (GB): 7.94531


In [ ]:
import torch
with torch.no_grad():
    torch.cuda.empty_cache()

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
print(isinstance(model, torch.nn.Module))
print(wikitext_batch_size)
print(model.config.max_position_embeddings)
print(tokenizer.model_max_length)

In [ ]:
import torch
import torchmetrics
import tqdm
from torch.cuda.amp import autocast, GradScaler

with torch.no_grad():
    torch.cuda.empty_cache()

def evaluate_perplexity_v2(model, dataloader, stride=512, max_length=None, device="cuda", to_device=False):
    if isinstance(model, torch.nn.Module):
        model.eval()
    if max_length is None:
        max_length = tokenizer.model_max_length
    if to_device:
        model.to(device)

    metric = torchmetrics.text.Perplexity(ignore_index=-100).to(device)  # -100 is the padding token.
    #!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
    
    for i, (x, y) in enumerate(dataloader):
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)
        
        with torch.no_grad() and autocast():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
            
            # import numpy as np
            # target_ids = x.clone().to(device)
            # target_ids[:, :-2048] = -100
            # outputs_v2 = model(x, labels=target_ids)
            # logits_v2 = outputs_v2["logits"][0].cpu().detach().numpy()
            # sequences = logits_v2.reshape(-1, 2048, logits_v2.shape[-1])
            # argmax_indices = np.argmax(sequences, axis=2)
            
            # Metric on current batch
            perplexity = metric(logits.float(), y)   
            print(f"Perplexity: {perplexity:.2f}")
            del outputs, logits, perplexity
            torch.cuda.empty_cache()

    # Metric on all batches using custom accumulation
    perplexity = metric.compute()
    torch.cuda.empty_cache()
    return perplexity.item()

wikitext_dataloader = wikitext_data_module.test_dataloader()
perpl = evaluate_perplexity_v2(model=model, dataloader=wikitext_dataloader, device="cuda")
print(perpl)

In [ ]:
perplexity

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
input_ids

In [ ]:
import torch
import tqdm

def evaluate_perplexity(model, tokenizer, data_module, data_split="validation", max_length=None, stride=512, factor=1, to_device=False, device="cuda"):
    model.eval()
    dataset_map = {
        "train": data_module.train_dataset,
        "validation": data_module.val_dataset,
        "test": data_module.test_dataset
    }
    
    if max_length is None:
        max_length = tokenizer.model_max_length
    if to_device:
        model.to(device)
        
    encodings = tokenizer("\n\n".join(dataset_map[data_split]["text"]), return_tensors="pt")
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end_loc = 0
    for begin_loc in tqdm.tqdm(range(0, seq_len//factor, stride)):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
            outputs = model(input_ids, labels=target_ids)
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'


            # loss is calculated using CrossEntropyLoss which averages over valid labels
            # N.B. the model only calculates loss over trg_len - 1 labels, because it internally shifts the labels
            # to the left by 1.
            neg_log_likelihood = outputs.loss
            del outputs

        nlls.append(neg_log_likelihood)
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).mean())
    print(f"Perplexity of model {model.NAME}: {ppl:.2f}")
    
    return ppl

evaluate_perplexity(model, tokenizer, wikitext_data_module, factor=10, device="cuda")

In [ ]:
evaluate_perplexity(model_bnb_8bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
awq_model.NAME = awq_model_name
evaluate_perplexity(awq_model, tokenizer, wikitext_data_module, to_device=True, device=device)

In [ ]:
print(len(tokenizer.vocab))
print(len(awq_tokenizer.vocab))

In [ ]:
evaluate_perplexity(awq_model, wikitext_dataloader, device="cuda")

In [ ]:
# Evaluate perplexity on each model
from src.evaluations.evaluate_text_generation import evaluate_perplexity

list_of_models = [model, model_bnb_8bit, model_bnb_4bit]
results = {}
for model in list_of_models:
  print(f"Perplexity for model {model.NAME}: {evaluate_perplexity(model_bnb_8bit, wikitext_data_module, device)}"

# Print perplexity results
#print(f"Perplexity (8-bit): {perplexity_8bit:.4f}")
#print(f"Perplexity (4-bit): {perplexity_4bit:.4f}")
print(f"Perplexity (Original): {perplexity_original:.4f}")

### 6.2. Brier Score

In [ ]:
import torch
import tqdm
import torch.nn.functional as F

def evaluate_brier_score(model, tokenizer, dataloader, max_length=None, stride=512, factor=1, to_device=False, device="cuda"):
    model.eval()
    
    if max_length is None:
        max_length = tokenizer.model_max_length
    if to_device:
        model.to(device)
        
    encodings = tokenizer("\n\n".join(dataloader.dataset.dataset["text"]), return_tensors="pt")
    seq_len = encodings.input_ids.size(1)

    brier_scores = []
    prev_end_loc = 0
    for begin_loc in tqdm.tqdm(range(0, seq_len//factor, stride)):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone().to(device)
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits
            
            # Shift logits and target_ids to the left by 1 for calculating the Brier score
            shifted_logits = logits[:, :-1].contiguous()
            shifted_target_ids = target_ids[:, 1:].contiguous()

            # Flatten the logits and target_ids for calculation
            shifted_logits = shifted_logits.view(-1, shifted_logits.size(-1))
            shifted_target_ids = shifted_target_ids.view(-1)

            # Filter out the -100 targets
            valid_indices = shifted_target_ids != -100
            valid_logits = shifted_logits[valid_indices]
            valid_target_ids = shifted_target_ids[valid_indices]

            # Get the probabilities
            probs = F.softmax(valid_logits, dim=-1)

            # Create one-hot target vectors
            targets = F.one_hot(valid_target_ids, num_classes=probs.size(-1)).float()

            # Calculate the Brier score
            brier_score = torch.mean((probs - targets) ** 2)
            brier_scores.append(brier_score)

    avg_brier_score = torch.stack(brier_scores).mean()
    print(f"Brier Score of model {model.NAME}: {avg_brier_score:.4f}")
    
    return avg_brier_score

In [ ]:
evaluate_brier_score(model, tokenizer, wikitext_dataloader, factor=100, device=device)

In [ ]:
evaluate_brier_score(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_brier_score(model_bnb_8bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_brier_score(awq_model, tokenizer, wikitext_data_module, device=device)

# 7. SEML Pipeline

In [ ]:
import shutil
import re
import os
import torch

# To avoid the following problem when running seml (see https://github.com/pytorch/pytorch/issues/37377)
os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
if re.match(".*username.*", os.getcwd()):
    CACHE_PATH = "~/.cache/"
else:
    CACHE_PATH = "/tmp/"

torch.hub.set_dir(CACHE_PATH)

import logging
logger = logging.getLogger("quant_logger")

#os.chdir('..')
print("Current Working Directory " , os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name
import seml

# Reload the src module after making changes
importlib.reload(src)

from src.models import get_model, get_model_name
from src.data import get_dataset, get_data_loader_from_split
from src.algorithms.quantization.quantize import quantize
from src.evaluations.evaluate_all import evaluate

In [ ]:
evaluate.__code__.co_varnames

In [ ]:
def run_quantize(
    # Dataset parameters
    seed_dataset=123,
    directory_dataset="",
    calib_dataset_name="",
    calib_dataset_split="",
    eval_dataset_name="",
    eval_dataset_split="",
    batch_size=1,
    stride=512,
    # Model parameters
    seed_model=123,
    directory_model="",
    clean_cache=True,
    model_name="",
    # Quantization parameters
    quantize_method="",
    quantize_params={},
    # Evaluation metrics
    eval_metrics=[
        "perplexity",
    ],
    device="cuda",
    save_quantized_model=False,
    quantized_model_save_path="",
):
    ##################
    ## Print config ##
    ##################
    logger.info("Received the following configuration:")
    logger.info(
        f"Calibration dataset: {calib_dataset_name}\n"
        f"Calibration split: {calib_dataset_split}\n"
        f"Evaluation dataset: {eval_dataset_name}\n"
        f"Evaluation split: {eval_dataset_split}\n"
        f"Batch size: {batch_size}\n"
        f"Model: {model_name}\n"
        f"Quantize method: {quantize_method}\n"
        f"Quantize params: {quantize_params}\n"
        f"Evaluation metrics: {eval_metrics}\n"
        f"Device: {device}\n"
    )
    
    ################
    ## Load model ##
    ################
    logger.info("Load base model")
    model_full_name = get_model_name(model_name)
    model, tokenizer = get_model(
        model_name=model_full_name,
        seed=seed_model,
        directory_model=directory_model,
        device=device,
    )
    
    ###############
    ## Load data ##
    ###############
    logger.info("Load calibration and evaluation data modules")
    calib_data_module = get_dataset(
        dataset_name=calib_dataset_name,
        directory_dataset=directory_dataset,
        batch_size=batch_size,
        sequence_length=stride,
        tokenizer_name=model_full_name,
        seed=seed_dataset,
    )
    eval_data_module = get_dataset(
        dataset_name=eval_dataset_name,
        directory_dataset=directory_dataset,
        batch_size=batch_size,
        sequence_length=stride,
        tokenizer_name=model_full_name,
        seed=seed_dataset,
    )
    
    calib_tokenizer = calib_data_module.tokenizer
    calib_dataloader = get_data_loader_from_split(calib_data_module, calib_dataset_split)
    
    eval_tokenizer = eval_data_module.tokenizer
    eval_dataloader = get_data_loader_from_split(eval_data_module, eval_dataset_split)

    ################################
    ## Update quantize parameters ##
    ################################
    logger.info("Defining quantize parameters")
    quantize_params.update(quantize_params)
    logger.info(f"Default parameters adjusted from {quantize_params}")

    ##############
    ## Quantize ##
    ##############
    logger.info("Quantization")
    quantized_model, quantized_tokenizer = quantize(
        model_name=model_full_name,
        calib_tokenizer=calib_tokenizer,
        calib_dataloader=calib_dataloader,
        quantize_method=quantize_method,
        quantize_config=quantize_params,
        save_model=save_quantized_model,
        save_path=quantized_model_save_path,
        device=device
    )

    ##############
    ## Evaluate ##
    ##############
    logger.info("Evaluating the quantized models")
    results = evaluate(
        model=quantized_model,
        eval_tokenizer=eval_tokenizer,
        eval_dataloader=eval_dataloader,
        eval_metrics=eval_metrics,
        stride=stride,
        factor=100,
        device=device,
        to_device=(quantize_method in ["AWQ"]),
        prefix=f"{model_name}_",
    )

    ####################
    ## Cleaning cache ##
    ####################
    logger.info
    if clean_cache:
        for root, dirs, files in os.walk(CACHE_PATH, topdown=False):
            for dir_name in dirs:
                pattern = re.compile(f"^.*{model_full_name.split('/')[-1]}.*")
                dir_path = os.path.join(root, dir_name)
                if re.match(pattern, dir_path):
                    try:
                        shutil.rmtree(dir_path)
                    except:
                        pass

    fail_trace = {
        "fail_trace": seml.evaluation.get_results,
    }

    return {**results, **fail_trace}


In [ ]:
import itertools
import random

# Fixed parameters
fixed_params = {
    'device': 'cuda',
    'clean_cache': True,
    'save_quantized_model': True,
    'seed_model': 123,
    'seed_dataset': 123,
    'batch_size': 1,
    'eval_metrics': ['perplexity', 'brier_score'],
    'calib_dataset_split': 'validation',
    'eval_dataset_split': 'test',
}

# Grid parameters
grid_params = {
    'calib_dataset_name': ['WikiText', 'OpenAssistant'],
    'eval_dataset_name': ['WikiText', 'OpenAssistant'],
    'quantize_method': ['BNB', 'AWQ'],
    'quantize_params': [
        {
            "num_bits": 8,
            "llm_int8_threshold": 6.0,
            "llm_int8_enable_fp32_cpu_offload": False,
            "llm_int8_has_fp16_weight": False
        },
        {
            "num_bits": 4,
            "bnb_4bit_compute_dtype": torch.bfloat16,
            "bnb_4bit_quant_type": "fp4",
            "bnb_4bit_use_double_quant": False,
        }
    ],
    'model_name': ['TinyLlama']
}

batch_sizes = [1]

# # Random parameters
# random_params = {
#     'samples': 2,
#     'seed': 12345,
#     'batch_size': {
#         'type': 'uniform',
#         'min': 1,
#         'max': 4
#     }
# }

# # Set seed for reproducibility
# random.seed(random_params['seed'])

# # Generate random batch sizes
# batch_sizes = [random.randint(random_params['batch_size']['min'], random_params['batch_size']['max']) for _ in range(random_params['samples'])]

# Generate all combinations for grid search
grid_combinations = list(itertools.product(
    grid_params['calib_dataset_name'],
    grid_params['eval_dataset_name'],
    grid_params['quantize_method'],
    grid_params['quantize_params'],
    grid_params['model_name']
))

# Run the quantize function for all combinations
results = []
max_combinations = 10000  # Set to a lower number for testing purposes
for i, combination in enumerate(grid_combinations):
    if i >= max_combinations:
        break
    for batch_size in batch_sizes:
        calib_dataset_name, eval_dataset_name, quantize_method, quantize_params, model_name = combination
        result = run_quantize(
            # Fixed parameters
            device=fixed_params['device'],
            clean_cache=fixed_params['clean_cache'],
            save_quantized_model=fixed_params['save_quantized_model'],
            seed_model=fixed_params['seed_model'],
            seed_dataset=fixed_params['seed_dataset'],
            eval_metrics=fixed_params['eval_metrics'],
            calib_dataset_split=fixed_params['calib_dataset_split'],
            eval_dataset_split=fixed_params['eval_dataset_split'],
            # Grid parameters
            calib_dataset_name=calib_dataset_name,
            eval_dataset_name=eval_dataset_name,
            quantize_method=quantize_method,
            quantize_params=quantize_params,
            model_name=model_name,
            # Random parameters
            batch_size=batch_size,
            stride=512,
            # Model parameters
            directory_model="",
            directory_dataset="",
            quantized_model_save_path=""
        )
        results.append(result)

# Do something with the results
print(results)


In [ ]:
print(wikitext_data_module.val_dataset["text"][:100])
print(wikitext_dataloader.dataset.dataset["text"][:100])

print(len(wikitext_data_module.val_dataset["text"]))
print(len(wikitext_dataloader.dataset.dataset["text"]))

In [ ]:
print(oasst_data_module.val_dataset["text"][:100])
print(oasst_dataloader.dataset["text"][:100])

print(len(oasst_data_module.val_dataset["text"]))
print(len(oasst_dataloader.dataset.dataset["text"]))

In [ ]:
print(wikitext_dataloader.dataset)
print(oasst_dataloader.dataset)

In [ ]:
results